# Notebook 01 — Embeddings, By Hand

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This notebook uses only Python's built-in `math` — there is nothing to install.

**The promise:** by the end, you can explain — to a friend, in 90 seconds, with no jargon —
what an embedding is.

We do not call any model or API here. We **build** tiny embeddings by hand for ten words,
write the similarity math ourselves, and watch the numbers match our intuition. The real
model comes in Notebook 03.

## The central analogy: a map of meaning

> Think of an embedding as **coordinates on a map of meaning.**
>
> On a real map, cities with similar latitude/longitude are close together. On a map of
> meaning, things with similar *meaning* are close together. `cat` sits near `dog`, not
> near `car`.
>
> The twist: this map has **more than two directions**. The real model we use later has
> **384** of them. They aren't labelled — the model invented them. But the *idea* is
> exactly the same as a 2-D map.

**Today we cheat:** we make a map with only **4 directions**, and we label them ourselves.
That's not how real embeddings work, but it shows the *shape* of how they work, in code you
can read top to bottom.

## Step 1 — One import

We use only `math` from Python's standard library. No frameworks today, on purpose, so
every line is readable.

When you run the next cell, nothing is calculated yet. You should just see a single line printed back to you confirming that `math` is loaded and we are ready to go. If you see that line with no red error text, the cell worked.

In [1]:
import math  # the only library we need today; it gives us square roots and more

# Print a short confirmation so you can see the cell ran successfully.
print("Ready — using only Python's built-in math today.")

Ready — using only Python's built-in math today.


## Step 2 — Build 4-D word vectors by hand

We'll place **ten words** in a 4-dimensional space. We pick the four directions by hand and
**name** them, so each vector is readable:

| Direction | Meaning | 0 means… | 1 means… |
|---|---|---|---|
| `animal_ness` | how animal-like | not an animal | definitely an animal |
| `vehicle_ness` | how vehicle-like | not a vehicle | definitely a vehicle |
| `size` | physical size | tiny | huge |
| `alive` | alive / moves on its own | inert | lively |

So `cat` scores high on `animal_ness`, low on `vehicle_ness`, small on `size`, high on
`alive`: `[0.9, 0.0, 0.2, 0.9]`. **Notice:** *someone* makes these numbers up. That's the
point — it shows where the numbers come from before a model hides that from us.

When you run the next cell, you should see a tidy table with one row per word and four number columns (animal, vehicle, size, alive), followed by a line confirming that ten words were placed in the space. The numbers in each row are the four coordinates we typed in by hand.

In [2]:
# Ten words, each a hand-built 4-D vector.
# The four numbers in every list are, in order:  animal  vehicle  size  alive.
#                       animal  vehicle  size   alive
word_vectors = {
    "cat":      [0.9,   0.0,   0.2,   0.9],   # small lively animal
    "dog":      [0.9,   0.0,   0.3,   0.9],   # like a cat, a touch bigger
    "lion":     [0.9,   0.0,   0.8,   0.9],   # a large animal
    "mouse":    [0.9,   0.0,   0.1,   0.9],   # a tiny animal
    "car":      [0.0,   0.9,   0.5,   0.7],   # a medium vehicle
    "truck":    [0.0,   0.9,   0.9,   0.7],   # a big vehicle
    "bicycle":  [0.0,   0.8,   0.2,   0.5],   # a small vehicle
    "airplane": [0.0,   0.9,   1.0,   0.8],   # a huge vehicle
    "tree":     [0.0,   0.0,   0.7,   0.6],   # not animal, not vehicle, fairly large
    "rock":     [0.0,   0.0,   0.5,   0.0],   # not animal, not vehicle, not alive
}

# Print them in a neat table.
# The :<10 and :>8 parts set column widths so the numbers line up.
print(f"{'word':<10} {'animal':>8} {'vehicle':>8} {'size':>6} {'alive':>6}")
print("-" * 42)  # a divider line, 42 dashes wide
# Walk through every word and its vector, printing one row each.
for word, vec in word_vectors.items():
    # vec[0]..vec[3] are the four numbers; .1f shows one decimal place.
    print(f"{word:<10} {vec[0]:>8.1f} {vec[1]:>8.1f} {vec[2]:>6.1f} {vec[3]:>6.1f}")

# len(word_vectors) counts how many words we stored (10 here).
print(f"\n{len(word_vectors)} words placed in a 4-D space.")

word         animal  vehicle   size  alive
------------------------------------------
cat             0.9      0.0    0.2    0.9
dog             0.9      0.0    0.3    0.9
lion            0.9      0.0    0.8    0.9
mouse           0.9      0.0    0.1    0.9
car             0.0      0.9    0.5    0.7
truck           0.0      0.9    0.9    0.7
bicycle         0.0      0.8    0.2    0.5
airplane        0.0      0.9    1.0    0.8
tree            0.0      0.0    0.7    0.6
rock            0.0      0.0    0.5    0.0

10 words placed in a 4-D space.


### Think about it

- `airplane` has `alive = 0.8`, not 0. Why might the table-builder do that? (Planes *move*
  and feel lively, even though they aren't alive.)
- `bicycle` has `vehicle_ness = 0.8`, not 0.9 like the car. Whoever builds the table
  decides. **The same word can be "close to a car" or "close to a horse" depending on what
  the directions mean.**

## Step 3 — Write the similarity math ourselves

Two words are "similar" if their vectors are similar. But what does that mean with numbers?
Two answers show up everywhere in embeddings.

### Dot product — "how much do these two agree?"

Multiply each matching pair of numbers, then add it all up. Big when both vectors are big
in the same directions.

$$\text{dot}(a, b) = \sum_i a_i \times b_i$$

### Cosine similarity — "are these two pointing the same way?"

Divide the dot product by the lengths of both vectors. This **cancels out length** — only
the *direction* (the angle between the arrows) matters. Cosine runs from -1 to +1; for our
all-positive vectors it stays in 0 to 1.

$$\cos(a, b) = \frac{\text{dot}(a, b)}{|a| \times |b|}$$

> Dot product: "cosine, but longer vectors win." Cosine: "ignore length, just compare
> direction." **For text, cosine is almost always what you want** — and it's the function we
> reuse for the rest of the course.

When you run the next cell, it defines the three functions and then does one quick check: the cosine of `cat` with itself. You should see a printed line reading `cosine(cat, cat) = 1.000`, then a note that the functions are defined. A vector is always perfectly similar to itself, so that 1.000 tells us the math is wired up correctly.

In [3]:
# Dot product: multiply matching numbers, then sum them.
def dot_product(a, b):
    # zip(a, b) pairs up the numbers: a[0] with b[0], a[1] with b[1], and so on.
    # We multiply each pair (x * y) and sum() adds all the products together.
    return sum(x * y for x, y in zip(a, b))

# Magnitude: the straight-line length of a vector.
def magnitude(v):
    # Square every number, add them up, then take the square root (Pythagoras in 4-D).
    return math.sqrt(sum(x * x for x in v))

# Cosine similarity: dot divided by both lengths — keeps direction, drops length.
def cosine_similarity(a, b):
    # Divide the dot product by the two magnitudes, which cancels out length.
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

# A vector compared with itself should score exactly 1.0.
cat = word_vectors["cat"]  # grab the cat vector to test with
# Compare cat with itself; :.3f rounds the result to three decimals.
print(f"cosine(cat, cat) = {cosine_similarity(cat, cat):.3f}   (should be 1.000)")
print("\nSimilarity functions defined.")

cosine(cat, cat) = 1.000   (should be 1.000)

Similarity functions defined.


## Step 4 — Predict, then verify

Before running the next cell, guess for each pair: will cosine be **high** (≥ 0.95),
**medium** (0.7–0.95), or **low** (< 0.7)?

| Pair | Your guess |
|---|---|
| `cat` ↔ `dog` | ? |
| `cat` ↔ `lion` | ? |
| `car` ↔ `truck` | ? |
| `cat` ↔ `car` | ? |
| `tree` ↔ `rock` | ? |
| `airplane` ↔ `bicycle` | ? |
| `mouse` ↔ `airplane` | ? |

When you run the next cell, you should see a table with one row per pair, showing the `dot` score and the `cosine` score side by side. Similar words (like `cat` and `dog`) land near 1.0; unrelated words (like `mouse` and `airplane`) land much lower. Now run it and see how your guesses did.

In [4]:
# The seven word pairs we want to compare, written as (word A, word B) tuples.
pairs = [
    ("cat",      "dog"),
    ("cat",      "lion"),
    ("car",      "truck"),
    ("cat",      "car"),
    ("tree",     "rock"),
    ("airplane", "bicycle"),
    ("mouse",    "airplane"),
]

# Print a header row, then a divider, so the numbers below line up in columns.
print(f"{'word A':<10} {'word B':<10} {'dot':>8}  {'cosine':>8}")
print("-" * 40)
# Go through each pair and compute both scores.
for a, b in pairs:
    # Look up the 4-D vector for each word in the pair.
    va, vb = word_vectors[a], word_vectors[b]
    # Print the two words, their dot product, and their cosine (both to 3 decimals).
    print(f"{a:<10} {b:<10} {dot_product(va, vb):>8.3f}  {cosine_similarity(va, vb):>8.3f}")

print("\nCompared.")

word A     word B          dot    cosine
----------------------------------------
cat        dog           1.680     0.997
cat        lion          1.780     0.919
car        truck         1.750     0.968
cat        car           0.730     0.455
tree       rock          0.350     0.759
airplane   bicycle       1.320     0.874
mouse      airplane      0.820     0.410

Compared.


### What to notice
1. How many did you get right? (Few people get all seven first try.)
2. Compare the `dot` and `cosine` columns. They often agree — but dot punishes short
   vectors, while cosine ignores length. Find a row where they disagree and see why.

## Step 5 — Nearest neighbours: "who hangs out with whom?"

Now flip the question. Instead of "how close are A and B?", ask: *for a given word, which
words are closest?* This is exactly what a vector database does — the heart of search.
We'll write it in five lines.

When you run the next cell, you should see three short lists: the top-3 closest words to `cat`, to `car`, and to `tree`, each with its cosine score in parentheses. Expect `cat` to be surrounded by other animals and `car` by other vehicles. `tree` is the odd one, and we look at why just below.

In [5]:
# For any word, list its k closest neighbours by cosine similarity.
def nearest(word, k=3):
    # Every other word except the one we're asking about (no point comparing to itself).
    others = [w for w in word_vectors if w != word]
    # Build a list of (other_word, similarity_score) pairs for each of those words.
    scored = [(w, cosine_similarity(word_vectors[word], word_vectors[w])) for w in others]
    # Sort by the score (pair[1]); reverse=True puts the highest similarity first.
    scored.sort(key=lambda pair: pair[1], reverse=True)
    # Return just the first k pairs, i.e. the k closest words.
    return scored[:k]

print("Top-3 closest words by cosine similarity:\n")
# Ask for the neighbours of three sample words and print each list.
for word in ["cat", "car", "tree"]:
    # Format each neighbour as 'word (score)' and join them with commas.
    neighbours = ", ".join(f"{w} ({s:.2f})" for w, s in nearest(word))
    print(f"  {word:<8} -> {neighbours}")

print("\nNearest-neighbour search in five lines.")

Top-3 closest words by cosine similarity:

  cat      -> dog (1.00), mouse (1.00), lion (0.92)
  car      -> bicycle (0.97), truck (0.97), airplane (0.96)
  tree     -> airplane (0.82), lion (0.79), truck (0.78)

Nearest-neighbour search in five lines.


### What just happened

`cat`'s neighbours are other small animals. `car`'s are other vehicles. And `tree`? Its
nearest neighbour turns out to be `airplane` — surprising, until you remember we made both
fairly *large*. A reminder that the directions *we chose* decide what "similar" means.
**This is search in miniature.** Notebook 03 does the same thing — but with 384-dimensional
vectors from a real model over real sentences. The mechanism is identical.

## Step 6 — Your turn: add a word

Below we add `whale` with hand-picked numbers (a huge ocean animal), then show its
neighbours. **Try it yourself:** change the four numbers, or add a different word
(`scooter`, `flower`, `helicopter`...). Predict its neighbours first, then run.

When you run the next cell, it adds `whale` to our collection and then prints `whale`'s three nearest neighbours with their cosine scores. Because we gave `whale` high `animal_ness` and `alive` values, expect other large, lively animals (like `lion`) to show up at the top.

In [6]:
# Add a new word by storing its 4 numbers:  animal  vehicle  size   alive
word_vectors["whale"] = [0.9,    0.0,    1.0,   0.9]   # huge, lively ocean animal

your_word = "whale"  # change this to any word you added above
# Print a heading naming the word we're inspecting.
print(f"Nearest neighbours of '{your_word}':")
# nearest() returns (word, score) pairs; print each on its own line.
for w, s in nearest(your_word):
    print(f"  {w:<10} {s:.3f}")

Nearest neighbours of 'whale':
  lion       0.995
  dog        0.907
  cat        0.873


## Recap

- **What is an embedding?** Coordinates on a map of meaning.
- **Why cosine over dot for text?** Cosine ignores length and compares direction.
- **How do you find similar items?** Cosine to each one, sort, take the top k — exactly what
  `nearest()` did.
- **Our 4-D vectors vs a real model's 384-D vectors?** Ours have human-named directions; the
  model's are learned from data and unreadable. The *shape* is the same.

**Next (Notebook 02):** we used 4 directions because we could read them. But how do you
*see* 4 (or 384) directions at once? That's PCA — squashing many directions into a 2-D
picture you can look at.

## Practice — Your Turn

Three short exercises to lock in what you just built. For each one, read the task,
make a quick prediction, then run the answer cell to check yourself. The answers reuse
the functions and the word table you already defined above, so run the earlier cells
first.

### Exercise 1 — Dot product by hand, then verify

Take two simple 2-D vectors: `(2, 3)` and `(1, 4)`.

Work the dot product out on paper first. Multiply the matching numbers and add them up:
`2 times 1` is `2`, and `3 times 4` is `12`, so the total should be `14`.

Now let `dot_product` do the same work and confirm it agrees with your number.

Try it yourself, then run the answer cell below.

In [7]:
# Answer
a = [2, 3]                      # our first vector, written as a 2-element list
b = [1, 4]                      # our second vector, same length

result = dot_product(a, b)      # reuse the dot_product function defined earlier
print(f"dot_product({a}, {b}) = {result}")   # show the computed value
print("By hand: 2*1 + 3*4 = 2 + 12 = 14")    # the matching by-hand calculation
print("Match:", result == 14)                 # True if the function agrees with our paper math

dot_product([2, 3], [1, 4]) = 14
By hand: 2*1 + 3*4 = 2 + 12 = 14
Match: True


### Exercise 2 — Same direction, different length

Compare `[1, 2]` with `[2, 4]`. The second vector is the first one doubled, so the two
arrows point in exactly the same direction. They differ only in length.

Cosine similarity ignores length and looks at direction, so predict the score: it should
come out to `1.0` (or as close as floating-point math allows).

Try it yourself, then run the answer cell below.

In [8]:
# Answer
short = [1, 2]                          # a short vector
long = [2, 4]                           # the same direction, twice as long

score = cosine_similarity(short, long)  # reuse cosine_similarity from earlier
print(f"cosine_similarity({short}, {long}) = {score:.3f}")  # round to 3 decimals for display

# Why does length not matter? Their lengths differ but the angle between them is zero.
print(f"length of {short} = {magnitude(short):.3f}")   # magnitude() shows they differ in length
print(f"length of {long}  = {magnitude(long):.3f}")
print("Same direction means cosine = 1.000, even though the lengths differ.")

cosine_similarity([1, 2], [2, 4]) = 1.000
length of [1, 2] = 2.236
length of [2, 4]  = 4.472
Same direction means cosine = 1.000, even though the lengths differ.


### Exercise 3 — Add a hamster and find its neighbours

Add a new word, `hamster`, to the table. A hamster is a small, lively animal, so give it
high `animal_ness`, zero `vehicle_ness`, a small `size`, and high `alive`:
`[0.9, 0.0, 0.1, 0.9]`.

Predict where it lands. Since those numbers look a lot like `cat` and `mouse`, expect its
nearest neighbours to be other small animals, not vehicles.

We add it to a copy of the table so the original `word_vectors` stays exactly as the
earlier cells left it. The answer defines a small helper that searches that copy.

Try it yourself, then run the answer cell below.

In [9]:
# Answer
practice_vectors = dict(word_vectors)            # a copy, so we never touch the original table
practice_vectors["hamster"] = [0.9, 0.0, 0.1, 0.9]  # small, lively animal: animal, vehicle, size, alive

# A nearest-neighbour search over our copy, same logic as the nearest() function above.
def nearest_in(table, word, k=3):
    others = [w for w in table if w != word]     # every word except the one we asked about
    # score each other word by cosine similarity to our target word
    scored = [(w, cosine_similarity(table[word], table[w])) for w in others]
    scored.sort(key=lambda pair: pair[1], reverse=True)  # highest similarity first
    return scored[:k]                            # keep only the top k

print("Nearest neighbours of 'hamster':")
for w, s in nearest_in(practice_vectors, "hamster"):  # walk the top-3 results
    print(f"  {w:<10} {s:.3f}")                  # print each neighbour and its score

# Confirm we did not change the original dictionary used in the earlier cells.
print("\nOriginal table still has", len(word_vectors), "words (hamster lives only in the copy).")

Nearest neighbours of 'hamster':
  mouse      1.000
  cat        0.997
  dog        0.988

Original table still has 11 words (hamster lives only in the copy).
